In [4]:
!pip install mlflow

  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached matplotlib-3.10.8-cp310-cp310-win_amd64.whl.metadata (52 kB)
  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp310-cp310-win_amd64.whl.metadata (2.8 kB)
  Using cached graphql_rela

In [6]:
!pip install nltk

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   -------------------- ------------------- 0.8/1.5 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 7.3 MB/s  0:00:00

   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ------------- -------------------------- 1/3 [regex]
   ------------- -------------------------- 1/3 [regex]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   ------

In [7]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [18]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')  # optional, but improves lemmatization

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\arupa\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\arupa\AppData\Roaming\nltk_data...


True

In [31]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
868,The movie is good and I think Tiffany Amber is...,positive
342,I see that C. Thomas Howell has appeared in ma...,negative
460,1989 was already a year in where Eddie Murphy ...,negative
609,1st watched 12/24/2009  4 out of 10 (Dir-Robe...,negative
708,I've been looking forward to the release of th...,positive


In [32]:
#data_preprocessing
def lemmatization(text):
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def remove_numbers(text):
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def remove_punctuations(text):
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def remove_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(remove_numbers)
        df['review'] = df['review'].apply(remove_punctuations)
        df['review'] = df['review'].apply(remove_urls)
        df['review'] = df['review'].apply(lemmatization)
        
        return df
    except Exception as e:
        print(f"Error during text normalization: {e}")
        raise

In [33]:
df = normalize_text(df)
df.head()

,review,sentiment
868,movie good think tiffany amber beautiful liked...,positive
342,see c thomas howell appeared many movie since ...,negative
460,already year eddie murphy longer hot started m...,negative
609,st watched  dir robert elli miller emotional ...,negative
708,looking forward release movie since first hear...,positive


In [34]:
df['sentiment'].value_counts()

sentiment
negative    256
positive    244
Name: count, dtype: int64

In [35]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [36]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df.head()

,review,sentiment
868,movie good think tiffany amber beautiful liked...,1
342,see c thomas howell appeared many movie since ...,0
460,already year eddie murphy longer hot started m...,0
609,st watched  dir robert elli miller emotional ...,0
708,looking forward release movie since first hear...,1


In [38]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [57]:
vectorizer = CountVectorizer(max_features=100)
x = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [58]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)

In [59]:
print(X_train.shape[0])
print(X_test.shape[0])

375
125


In [42]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/ArupaBarua/Capstone-Project.mlflow')
dagshub.init(repo_owner='ArupaBarua', repo_name='Capstone-Project', mlflow=True)

mlflow.set_experiment("Logistic Regression Baseline")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\arupa\.conda\envs\atlas\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=7fdc83b2-630b-47a4-b98a-cdc39f1c6082&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=060d2dc5bb6c5ce46a38f105574004b344329b33a40cf83565be755395b6b4df




Accessing as ArupaBarua

Initialized MLflow to track repo "ArupaBarua/Capstone-Project"

Repository ArupaBarua/Capstone-Project initialized!

2026/03/11 17:19:29 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/4b2be7a7ecbb4c0a85d7b3a9537d7f11', creation_time=1773227971168, experiment_id='0', last_update_time=1773227971168, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, workspace='default'>

In [60]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)
        raise


2026-03-11 18:01:26,890 - INFO - Starting MLflow run...
2026-03-11 18:01:27,467 - INFO - Logging preprocessing parameters...
2026-03-11 18:01:28,521 - INFO - Initializing Logistic Regression model...
2026-03-11 18:01:28,522 - INFO - Fitting the model...
2026-03-11 18:01:28,559 - INFO - Model training complete.
2026-03-11 18:01:28,560 - INFO - Logging model parameters...
2026-03-11 18:01:28,955 - INFO - Making predictions...
2026-03-11 18:01:28,956 - INFO - Calculating evaluation metrics...
2026-03-11 18:01:28,972 - INFO - Logging evaluation metrics...
2026-03-11 18:01:30,525 - INFO - Saving and logging the model...
2026/03/11 18:01:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/11 18:01:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run welcoming-skunk-315 at: https://dagshub.com/ArupaBarua/Capstone-Project.mlflow/#/experiments/0/runs/a21163c6cf4248748f1441d641c123fc
🧪 View experiment at: https://dagshub.com/ArupaBarua/Capstone-Project.mlflow/#/experiments/0
